In [ ]:
# NLP Lab - Week 1: Text Foundations

Topics: Tokenization, Stemming & Lemmatization, Stop Words, Bag of Words & N-grams, TF-IDF, String Similarity

In [7]:
# Imports and NLP resource downloads and Installing required packages

import subprocess, sys

LIBS = ["nltk", "spacy", "scikit-learn", "transformers", "Levenshtein", "tabulate", "pandas", "numpy"]

for lib in LIBS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", lib, "-q"])

import re, math, string, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import nltk
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet",
                 "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.corpus import stopwords as nltk_stopwords

import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], capture_output=True)
    nlp = spacy.load("en_core_web_sm")

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
import Levenshtein as lev


#Section 1: Tokenization

#Tokenization splits raw text into tokens (words, sub-words, or characters). It is the first step of almost every NLP pipeline. There is no single best tokenizer - the right choice depends on the language, domain, and task.

# Compare different tokenization methods on tricky inputs
sample_texts = [
    "Dr. Smith couldn't attend the U.N. meeting -- it's a shame!",
    "I love NLP!!! #machinelearning @OpenAI https://huggingface.co",
    "The price is $4.99 and the date is 22/05/2025.",
]

# Whitespace
def whitespace_tokenize(text):
    return text.split()

# Regex (words + numbers)
def regex_tokenize(text):
    pattern = r"\b[a-zA-Z']+\b|\d+(?:\.\d+)?"
    return re.findall(pattern, text)

# NLTK
def nltk_tokenize(text):
    return word_tokenize(text)

# spaCy
def spacy_tokenize(text):
    doc = nlp(text)
    return [token.text for token in doc]

# GPT-2 BPE (sub-word)
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
def bpe_tokenize(text):
    return gpt2_tok.tokenize(text)

methods = {
    "Whitespace": whitespace_tokenize,
    "Regex": regex_tokenize,
    "NLTK": nltk_tokenize,
    "spaCy": spacy_tokenize,
    "GPT-2 BPE": bpe_tokenize,
}

for i, text in enumerate(sample_texts):
    print(f"\n{'='*72}")
    print(f"Text {i+1}: {text}")
    print(f"{'='*72}")
    for name, fn in methods.items():
        tokens = fn(text)
        print(f"[{name:7s}] ({len(tokens):2d} tokens)  {tokens}")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


Text 1: Dr. Smith couldn't attend the U.N. meeting -- it's a shame!
[Whitespace] (11 tokens)  ['Dr.', 'Smith', "couldn't", 'attend', 'the', 'U.N.', 'meeting', '--', "it's", 'a', 'shame!']
[Regex  ] (11 tokens)  ['Dr', 'Smith', "couldn't", 'attend', 'the', 'U', 'N', 'meeting', "it's", 'a', 'shame']
[NLTK   ] (14 tokens)  ['Dr.', 'Smith', 'could', "n't", 'attend', 'the', 'U.N.', 'meeting', '--', 'it', "'s", 'a', 'shame', '!']
[spaCy  ] (14 tokens)  ['Dr.', 'Smith', 'could', "n't", 'attend', 'the', 'U.N.', 'meeting', '--', 'it', "'s", 'a', 'shame', '!']
[GPT-2 BPE] (18 tokens)  ['Dr', '.', 'ĠSmith', 'Ġcouldn', "'t", 'Ġattend', 'Ġthe', 'ĠU', '.', 'N', '.', 'Ġmeeting', 'Ġ--', 'Ġit', "'s", 'Ġa', 'Ġshame', '!']

Text 2: I love NLP!!! #machinelearning @OpenAI https://huggingface.co
[Whitespace] ( 6 tokens)  ['I', 'love', 'NLP!!!', '#machinelearning', '@OpenAI', 'https://huggingface.co']
[Regex  ] ( 8 tokens)  ['I', 'love', 'NLP', 'machinelearning', 'OpenAI', 'https', 'huggingface', 'co']
[NLT

In [8]:
# Sentence tokenization
paragraph = """
Natural language processing is a subfield of linguistics and AI.
It focuses on the interactions between computers and human language.
NLP tasks include translation, summarization, and sentiment analysis.
Dr. Alan Turing proposed the Turing Test in 1950. Today, LLMs like GPT-4 pass it.
"""

sentences = sent_tokenize(paragraph)
for i, s in enumerate(sentences, start=1):
    print(f"Sentence {i}: {s.strip()}")
print(f"\nTotal: {len(sentences)} sentences")

Sentence 1: Natural language processing is a subfield of linguistics and AI.
Sentence 2: It focuses on the interactions between computers and human language.
Sentence 3: NLP tasks include translation, summarization, and sentiment analysis.
Sentence 4: Dr. Alan Turing proposed the Turing Test in 1950.
Sentence 5: Today, LLMs like GPT-4 pass it.

Total: 5 sentences


In [9]:
# Social-media tokenizer (keeps hashtags, mentions, URLs whole)
tweet = "Loving #NLP today @Stanford! Check https://arxiv.org for the latest papers"

def social_media_tokenize(text):
    pattern = r"#\w+|@\w+|https?://\S+|\b\w+\b"
    return re.findall(pattern, text)

tokens = social_media_tokenize(tweet)
print("Input :", tweet)
print("Tokens:", tokens)

Input : Loving #NLP today @Stanford! Check https://arxiv.org for the latest papers
Tokens: ['Loving', '#NLP', 'today', '@Stanford', 'Check', 'https://arxiv.org', 'for', 'the', 'latest', 'papers']


In [10]:
# Sub-word tokenization: BERT (WordPiece) vs GPT-2 (BPE)
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
medical_text = "The patient has hepatosplenomegaly and thrombocytopenia."

bert_tokens = bert_tok.tokenize(medical_text)
gpt2_tokens = gpt2_tok.tokenize(medical_text)

print("Input:", medical_text)
print(f"\nBERT  ({len(bert_tokens):2d} tokens): {bert_tokens}")
print(f"GPT-2 ({len(gpt2_tokens):2d} tokens): {gpt2_tokens}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Input: The patient has hepatosplenomegaly and thrombocytopenia.

BERT  (19 tokens): ['the', 'patient', 'has', 'he', '##pa', '##tos', '##ple', '##no', '##me', '##gal', '##y', 'and', 'th', '##rom', '##bo', '##cy', '##top', '##enia', '.']
GPT-2 (19 tokens): ['The', 'Ġpatient', 'Ġhas', 'Ġhepat', 'os', 'pl', 'en', 'ome', 'g', 'aly', 'Ġand', 'Ġth', 'rom', 'b', 'ocy', 'top', 'en', 'ia', '.']


In [11]:
#Section 2: Stemming & Lemmatization

#Both normalise words to a common root so "run", "running", and "ran" are treated the same. Stemming strips suffixes with rules (fast, can produce non-words). Lemmatization uses a dictionary + part-of-speech (accurate, always real words).

# Compare stemmers and lemmatizer side by side
porter = PorterStemmer()
snowball = SnowballStemmer("english")
lemmatize = WordNetLemmatizer()

words = [
    "running", "runs", "ran",
    "studies", "studying",
    "better", "good", "best",
    "universities",
    "happily", "happiness",
]

header = f"{'Word':<15} {'Porter':<12} {'Snowball':<12} {'Lemma(v)':<14} {'Lemma(n)'}"
print(header)
print("-" * len(header))

for word in words:
    p = porter.stem(word)
    s = snowball.stem(word)
    lv = lemmatize.lemmatize(word, pos="v")
    ln = lemmatize.lemmatize(word, pos="n")
    print(f"{word:<15} {p:<12} {s:<12} {lv:<14} {ln}")

Word            Porter       Snowball     Lemma(v)       Lemma(n)
-----------------------------------------------------------------
running         run          run          run            running
runs            run          run          run            run
ran             ran          ran          run            ran
studies         studi        studi        study          study
studying        studi        studi        study          studying
better          better       better       better         better
good            good         good         good           good
best            best         best         best           best
universities    univers      univers      universities   university
happily         happili      happili      happily        happily
happiness       happi        happi        happiness      happiness


In [12]:
# Normalize a product review: stems vs lemmas
review = "The students were studying really hard because their studies matter for their future careers."

tokens = word_tokenize(review.lower())
print("Tokens  :", tokens)

stems = [porter.stem(t) for t in tokens]
lemmas = [lemmatize.lemmatize(t) for t in tokens]

print("\nStems   :", stems)
print("Lemmas  :", lemmas)
print(f"\nUnique stems  : {len(set(stems))}")
print(f"Unique lemmas : {len(set(lemmas))}")

# Stemming gives fewer unique forms but can produce non-words,
# so it is not always better when meaning matters

Tokens  : ['the', 'students', 'were', 'studying', 'really', 'hard', 'because', 'their', 'studies', 'matter', 'for', 'their', 'future', 'careers', '.']

Stems   : ['the', 'student', 'were', 'studi', 'realli', 'hard', 'becaus', 'their', 'studi', 'matter', 'for', 'their', 'futur', 'career', '.']
Lemmas  : ['the', 'student', 'were', 'studying', 'really', 'hard', 'because', 'their', 'study', 'matter', 'for', 'their', 'future', 'career', '.']

Unique stems  : 13
Unique lemmas : 14


In [13]:
# spaCy POS-aware lemmatization vs NLTK (no POS hint)
sentence = "The geese were flying south while the mice ran through the better fields."
doc = nlp(sentence)

print(f"{'Token':<15} {'spaCy Lemma':<15} {'POS':<10} {'NLTK Lemma (no POS)'}")
print("-" * 62)

for token in doc:
    spacy_lemma = token.lemma_
    pos = token.pos_
    nltk_lemma = lemmatize.lemmatize(token.text.lower())
    print(f"{token.text:<15} {spacy_lemma:<15} {pos:<10} {nltk_lemma}")

# Without POS, NLTK leaves "ran", "flying", "better" unchanged; spaCy resolves them correctly

Token           spaCy Lemma     POS        NLTK Lemma (no POS)
--------------------------------------------------------------
The             the             DET        the
geese           goose           NOUN       goose
were            be              AUX        were
flying          fly             VERB       flying
south           south           ADV        south
while           while           SCONJ      while
the             the             DET        the
mice            mouse           NOUN       mouse
ran             run             VERB       ran
through         through         ADP        through
the             the             DET        the
better          well            ADJ        better
fields          field           NOUN       field
.               .               PUNCT      .


In [14]:
#Section 3: Stop Words, Punctuation & Case Normalization

#Stop words are high-frequency, low-information words ("the", "is", "at"). Removing them reduces noise for classification and retrieval, but it is NOT always correct - sentiment analysis needs "not"/"never" and NER needs words like "The".
#Pipeline: lowercase -> remove URLs -> remove punctuation -> tokenize -> remove stop words


# Five-step text cleaning pipeline
STOP_WORDS = set(nltk_stopwords.words("english"))
print(f"NLTK stop-word list: {len(STOP_WORDS)} words")
print("First 20:", sorted(STOP_WORDS)[:20])

def clean_text(text, remove_stops=True, verbose=False):
    if verbose: print(f"[0] original  : {text}")

    # lowercase
    text = text.lower()
    if verbose: print(f"[1] lowercase : {text}")

    # remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    if verbose: print(f"[2] no URLs   : {text}")

    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    if verbose: print(f"[3] no punct  : {text}")

    # tokenize
    tokens = word_tokenize(text)
    if verbose: print(f"[4] tokenized : {tokens}")

    # remove stop words and single chars
    if remove_stops:
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
        if verbose: print(f"[5] no stops  : {tokens}")

    return tokens

demo = "The NLP model at https://huggingface.co is NOT working correctly!!!"
result = clean_text(demo, verbose=True)
print(f"\nFinal tokens: {result}")
# Note: 'not' got removed as a stop word, which can flip sentiment meaning

NLTK stop-word list: 198 words
First 20: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
[0] original  : The NLP model at https://huggingface.co is NOT working correctly!!!
[1] lowercase : the nlp model at https://huggingface.co is not working correctly!!!
[2] no URLs   : the nlp model at  is not working correctly!!!
[3] no punct  : the nlp model at  is not working correctly
[4] tokenized : ['the', 'nlp', 'model', 'at', 'is', 'not', 'working', 'correctly']
[5] no stops  : ['nlp', 'model', 'working', 'correctly']

Final tokens: ['nlp', 'model', 'working', 'correctly']


In [15]:
# Apply the pipeline to a corpus and measure token reduction
corpus = [
    "I absolutely LOVE this product! It's amazing and works perfectly.",
    "This is the worst thing I've ever bought. Do NOT recommend it at all!",
    "Okay product, nothing special. The delivery was fast. Visit www.shop.com",
    "Can't believe how good this is!!! Best purchase of 2025... 10/10 would buy again.",
]

print(f"{'#':<3} {'Raw':>5} {'Clean':>6} {'Reduction':>10}   Cleaned tokens")
print("-" * 70)

for i, doc in enumerate(corpus, start=1):
    raw_n = len(word_tokenize(doc))
    clean_tokens = clean_text(doc)
    clean_n = len(clean_tokens)
    pct = (1 - clean_n/raw_n)*100 if raw_n else 0
    print(f"{i:<3} {raw_n:>5} {clean_n:>6} {pct:>9.0f}%   {clean_tokens}")

#     Raw  Clean  Reduction   Cleaned tokens
----------------------------------------------------------------------
1      13      6        54%   ['absolutely', 'love', 'product', 'amazing', 'works', 'perfectly']
2      17      6        65%   ['worst', 'thing', 'ive', 'ever', 'bought', 'recommend']
3      13      7        46%   ['okay', 'product', 'nothing', 'special', 'delivery', 'fast', 'visit']
4      20      9        55%   ['cant', 'believe', 'good', 'best', 'purchase', '2025', '1010', 'would', 'buy']


In [16]:
# Domain-specific stop words (medical)
medical_stops = STOP_WORDS | {
    "patient", "doctor", "prescribed", "reported", "noted", "presented", "admitted",
}

medical_notes = [
    "The patient presented with fever. The doctor had prescribed antibiotics.",
    "Patient has reported chest pain. Doctor ordered him an ECG and blood tests.",
    "The patient was discharged too. The doctor wrote a follow-up prescription.",
]

for note in medical_notes:
    tokens = word_tokenize(note.lower())
    filtered = [t for t in tokens if t not in medical_stops and len(t) > 1]
    print(f"Original : {note}")
    print(f"Filtered : {filtered}\n")

Original : The patient presented with fever. The doctor had prescribed antibiotics.
Filtered : ['fever', 'antibiotics']

Original : Patient has reported chest pain. Doctor ordered him an ECG and blood tests.
Filtered : ['chest', 'pain', 'ordered', 'ecg', 'blood', 'tests']

Original : The patient was discharged too. The doctor wrote a follow-up prescription.
Filtered : ['discharged', 'wrote', 'follow-up', 'prescription']



In [17]:
#Section 4: Bag of Words & N-grams

#Bag of Words represents a document as a count vector over a fixed vocabulary. Word order is discarded - only frequencies matter. Fast and interpretable, but no semantics and high-dimensional on large corpora. N-grams (e.g. bigrams) capture some local word order.

# Bag of Words with CountVectorizer
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog are friends",
    "cats and dogs never agree",
]

cv = CountVectorizer()
X = cv.fit_transform(corpus)

df_sk = pd.DataFrame(
    X.toarray(),
    columns=cv.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("CountVectorizer matrix:")
print(df_sk.to_string())

CountVectorizer matrix:
       agree  and  are  cat  cats  dog  dogs  friends  log  mat  never  on  sat  the
Doc 1      0    0    0    1     0    0     0        0    0    1      0   1    1    2
Doc 2      0    0    0    0     0    1     0        0    1    0      0   1    1    2
Doc 3      0    1    1    1     0    1     0        1    0    0      0   0    0    2
Doc 4      1    1    0    0     1    0     1        0    0    0      1   0    0    0


In [18]:
# Unigram vs bigram features
sentiment_docs = [
    "the movie was not good at all",
    "the movie was very good indeed",
    "not bad at all for a cheap movie",
]

uni_vec = CountVectorizer(ngram_range=(1, 1))
X_uni = uni_vec.fit_transform(sentiment_docs)
print("Unigram features:", uni_vec.get_feature_names_out().tolist())

bi_vec = CountVectorizer(ngram_range=(2, 2))
X_bi = bi_vec.fit_transform(sentiment_docs)
print("Bigram features:", bi_vec.get_feature_names_out().tolist())

# With unigrams "not good" and "very good" share the word "good", so they look similar.
# Bigrams capture "not good" vs "very good" directly, so a classifier can tell them apart

Unigram features: ['all', 'at', 'bad', 'cheap', 'for', 'good', 'indeed', 'movie', 'not', 'the', 'very', 'was']
Bigram features: ['all for', 'at all', 'bad at', 'cheap movie', 'for cheap', 'good at', 'good indeed', 'movie was', 'not bad', 'not good', 'the movie', 'very good', 'was not', 'was very']


In [19]:
# Term frequency analysis
news_corpus = [
    "Scientists discover new AI model that beats humans at chess and go",
    "New AI research shows large language models can reason about math",
    "Humans and AI collaborate on scientific discovery in new study",
    "Chess champion defeats AI model in controversial match",
    "Language models show surprising reasoning about human emotions",
]

vec = CountVectorizer(stop_words="english")
X = vec.fit_transform(news_corpus)
feature_names = vec.get_feature_names_out()
freq = X.toarray().sum(axis=0)
doc_freq = (X.toarray() > 0).sum(axis=0)

print("Q1 - Top 10 most frequent terms:")
top10 = np.argsort(freq)[::-1][:10]
for idx in top10:
    print(f"  {feature_names[idx]:<20} freq={freq[idx]}  doc_freq={doc_freq[idx]}")

print("\nQ2 - Terms in ALL 5 documents:")
print(feature_names[doc_freq == len(news_corpus)])

print("\nQ3 - Terms in exactly ONE document:")
print(feature_names[doc_freq == 1])

Q1 - Top 10 most frequent terms:
  ai                   freq=4  doc_freq=4
  new                  freq=3  doc_freq=3
  chess                freq=2  doc_freq=2
  models               freq=2  doc_freq=2
  model                freq=2  doc_freq=2
  language             freq=2  doc_freq=2
  humans               freq=2  doc_freq=2
  human                freq=1  doc_freq=1
  beats                freq=1  doc_freq=1
  champion             freq=1  doc_freq=1

Q2 - Terms in ALL 5 documents:
[]

Q3 - Terms in exactly ONE document:
['beats' 'champion' 'collaborate' 'controversial' 'defeats' 'discover'
 'discovery' 'emotions' 'human' 'large' 'match' 'math' 'reason'
 'reasoning' 'research' 'scientific' 'scientists' 'shows' 'study'
 'surprising']


In [20]:
#Section 5: TF-IDF

#TF-IDF gives high weight to words that are frequent in one document but rare across the corpus - the words that identify a document's topic.


#TF(t,d)= count(t in d) / total_words_in_d
#IDF(t)= log(N / df(t))
#TF-IDF(t,d)= TF(t,d) * IDF(t)


# TF-IDF with sklearn
docs = [
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks for machine learning",
    "natural language processing is a field of artificial intelligence",
    "neural networks are the backbone of deep learning models"
]

tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(docs)

df_tfidf = pd.DataFrame(
    X_tfidf.toarray().round(3),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(docs))]
)

print("TF-IDF matrix:")
print(df_tfidf.to_string())

# Top-3 terms per document
print("\nTop-3 most important terms per document:")
for i in range(len(docs)):
    top3 = df_tfidf.iloc[i].nlargest(3)
    print(f"  Doc {i+1}: {list(zip(top3.index, top3.round(3).values))}")

TF-IDF matrix:
         are  artificial  backbone   deep  field    for  intelligence     is  language  learning  machine  models  natural  networks  neural     of  processing  subset    the   uses
Doc 1  0.000       0.380     0.000  0.000  0.000  0.000         0.380  0.380     0.000     0.308    0.380   0.000    0.000     0.000   0.000  0.308       0.000   0.482  0.000  0.000
Doc 2  0.000       0.000     0.000  0.319  0.000  0.404         0.000  0.000     0.000     0.516    0.319   0.000    0.000     0.319   0.319  0.000       0.000   0.000  0.000  0.404
Doc 3  0.000       0.315     0.000  0.000  0.399  0.000         0.315  0.315     0.399     0.000    0.000   0.000    0.399     0.000   0.000  0.255       0.399   0.000  0.000  0.000
Doc 4  0.387       0.000     0.387  0.305  0.000  0.000         0.000  0.000     0.000     0.247    0.000   0.387    0.000     0.305   0.305  0.247       0.000   0.000  0.387  0.000

Top-3 most important terms per document:
  Doc 1: [('subset', np.float64(0

In [21]:
# Mini search engine using TF-IDF + cosine similarity
article_corpus = [
    "Python is a popular programming language used in data science and machine learning.",
    "Machine learning algorithms learn patterns from data without explicit programming.",
    "Natural language processing enables computers to understand human language.",
    "Deep learning neural networks have revolutionized computer vision and speech recognition.",
    "Data science combines statistics, programming, and domain expertise.",
]

queries = [
    "how do computers understand text?",
    "what programming language for data science?",
    "neural networks for images",
]

# fit on the corpus once
tfidf = TfidfVectorizer(stop_words="english")
corpus_matrix = tfidf.fit_transform(article_corpus)

for query in queries:
    query_vec = tfidf.transform([query])           # transform only, same vocabulary
    scores = cosine_similarity(query_vec, corpus_matrix)[0]
    best_idx = np.argmax(scores)
    print(f"Query : '{query}'")
    print(f"Match : '{article_corpus[best_idx]}'  (score={scores[best_idx]:.3f})")
    print()

# Use transform() (not fit_transform) on the query so it stays in the corpus vocabulary

Query : 'how do computers understand text?'
Match : 'Natural language processing enables computers to understand human language.'  (score=0.482)

Query : 'what programming language for data science?'
Match : 'Python is a popular programming language used in data science and machine learning.'  (score=0.591)

Query : 'neural networks for images'
Match : 'Deep learning neural networks have revolutionized computer vision and speech recognition.'  (score=0.487)



In [22]:
## Section 6: String Similarity Measures

#Edit(Levenshtein) distance: character level, min insert/delete/replace operations. Lower = more similar. Good for spell-check.
#Jaccard: word-set overlap, |A and B| / |A or B|. Range 0-1, higher = more similar.
#Cosine: angle between TF-IDF vectors. Range 0-1, higher = more similar. Ignores word order.


# Three similarity measures + comparison table
def edit_dist(s1, s2):
    return lev.distance(s1, s2)

def jaccard(s1, s2):
    a, b = set(s1.lower().split()), set(s2.lower().split())
    return len(a & b) / len(a | b) if a | b else 0

def cosine_sim(s1, s2):
    v = TfidfVectorizer()
    mat = v.fit_transform([s1, s2])
    return cosine_similarity(mat[0], mat[1])[0][0]

pairs = [
    ("kitten", "sitting"),
    ("hello world", "hello NLP"),
    ("I love NLP", "NLP I love"),
    ("data science", "data engineering"),
    ("the quick brown fox", "the slow green cat"),
]

print(f"{'Pair':<44} {'Edit':>6} {'Jaccard':>8} {'Cosine':>8}")
print("-" * 72)
for s1, s2 in pairs:
    label = f'"{s1}" vs "{s2}"'
    print(f"{label:<44} {edit_dist(s1, s2):>6} {jaccard(s1, s2):>8.3f} {cosine_sim(s1, s2):>8.3f}")

# "I love NLP" vs "NLP I love": cosine = 1.0 (same words) but edit distance is large
# because edit distance is sensitive to character position / order

Pair                                           Edit  Jaccard   Cosine
------------------------------------------------------------------------
"kitten" vs "sitting"                             3    0.000    0.000
"hello world" vs "hello NLP"                      5    0.333    0.336
"I love NLP" vs "NLP I love"                      8    1.000    1.000
"data science" vs "data engineering"              9    0.333    0.336
"the quick brown fox" vs "the slow green cat"     11    0.143    0.144


In [23]:
# Spelling corrector using edit distance
NLP_DICT = [
    "natural", "language", "processing", "machine", "learning",
    "algorithm", "tokenization", "lemmatization", "embedding", "transformer",
    "attention", "neural", "network", "classification", "regression",
]

misspelled = ["natral", "laguage", "procssing", "alogithm", "tokeniaztion"]

def find_closest(word, dictionary):
    return min(dictionary, key=lambda w: edit_dist(word, w))

for mw in misspelled:
    correction = find_closest(mw, NLP_DICT)
    print(f"  '{mw}'  ->  '{correction}'")

  'natral'  ->  'natural'
  'laguage'  ->  'language'
  'procssing'  ->  'processing'
  'alogithm'  ->  'algorithm'
  'tokeniaztion'  ->  'tokenization'


In [24]:
# Near-duplicate review detector (bonus)
reviews = [
    "Great product, very happy with my purchase!",
    "Great product, I am very happy with this purchase!",
    "Terrible quality, waste of money. Would not buy again.",
    "Awful quality, total waste of money. Never buying again.",
    "Fast delivery and good packaging. Product works fine.",
    "Quick shipping, nice box. Works as expected.",
]

THRESHOLD = 0.4

print(f"{'Pair':<8} {'Jaccard':>9} {'Cosine':>9} {'Flag':>12}")
print("-" * 45)
for i in range(len(reviews)):
    for j in range(i+1, len(reviews)):
        jac = jaccard(reviews[i], reviews[j])
        cos = cosine_sim(reviews[i], reviews[j])
        is_dup = "DUP" if jac > THRESHOLD and cos > THRESHOLD else ""
        print(f"({i},{j})   {jac:>9.3f} {cos:>9.3f} {is_dup:>12}")

# A threshold around 0.4-0.5 catches the true duplicates without many false positives.
# Cosine tends to run higher than Jaccard because it uses TF-IDF weights instead of plain overlap

Pair       Jaccard    Cosine         Flag
---------------------------------------------
(0,1)       0.600     0.674          DUP
(0,2)       0.000     0.000             
(0,3)       0.000     0.000             
(0,4)       0.000     0.072             
(0,5)       0.000     0.000             
(1,2)       0.000     0.000             
(1,3)       0.000     0.000             
(1,4)       0.000     0.067             
(1,5)       0.000     0.000             
(2,3)       0.385     0.388             
(2,4)       0.000     0.000             
(2,5)       0.000     0.000             
(3,4)       0.000     0.000             
(3,5)       0.000     0.000             
(4,5)       0.071     0.072             
